<a href="https://colab.research.google.com/github/sidhu2690/MARL/blob/main/Trust_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
import numpy as np
import torch
import torch.nn as nn

GRID = 12
N_ROBOTS = 12
ADV_FRAC = 0.34
STEPS = 150
EPSILON = 0.05
EPISODES = 600
LR = 2e-3
SEED = 0

rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

def make_true_map(grid):
  xs, ys = np.meshgrid(np.arange(grid), np.arange(grid))
  field = np.zeros((grid, grid))
  for _ in range(5):
    cx, cy = rng.uniform(0, grid, 2)
    amp = rng.uniform(0.5, 1.0)
    sigma = rng.uniform(1.5, 3.5)
    field += amp * np.exp(-((xs - cx) ** 2 + (ys - cy) ** 2) / (2 * sigma ** 2))
  field = (field - field.min()) / (field.max() - field.min() + 1e-8)
  return field

G = make_true_map(GRID)

n_adv = int(N_ROBOTS * ADV_FRAC)
is_adversarial = np.array([False] * (N_ROBOTS - n_adv) + [True] * n_adv)
rng.shuffle(is_adversarial)

def random_walk(grid, steps):
  pos = rng.integers(0, grid, size=2)
  cells = []
  for _ in range(steps):
    while True:
      move = rng.choice([-1, 0, 1], size=2)
      new_pos = np.clip(pos + move, 0, grid - 1)
      if not np.array_equal(new_pos, pos):
        break
    pos = new_pos
    cells.append(tuple(pos))
  return cells

In [ ]:
class AlphaNet(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(3, 16),
        nn.ReLU(),
        nn.Linear(16, 8),
        nn.ReLU(),
        nn.Linear(8, 1),
        nn.Sigmoid(),
        )

  def forward(self, feats):
    return self.net(feats).squeeze(-1)


alpha_net = AlphaNet()
optimizer = torch.optim.Adam(alpha_net.parameters(), lr=LR)

In [20]:
CONTRA_THRESH = 0.2

def report_value(true_val, agent_idx):
  if is_adversarial[agent_idx]:
    val = true_val + rng.choice([-1, 1]) * rng.uniform(0.3, 0.6)
    val += rng.normal(0, 0.1)
  else:
    val = true_val + rng.normal(0, 0.02)
  return float(np.clip(val, 0, 1))


In [40]:
loss_history = []

for ep in range(EPISODES):
  walks = [random_walk(GRID, STEPS) for _ in range(N_ROBOTS)]
  c = np.zeros(N_ROBOTS)
  D_sum = np.zeros(N_ROBOTS)
  D_n = np.zeros(N_ROBOTS)
  cell_reports = {}

  ep_loss = 0.0

  for t in range(STEPS):
    touched_cells = set()

    for i in range(N_ROBOTS):
      cell = walks[i][t]
      r_i = report_value(G[cell], i)

      others = cell_reports.get(cell, [])
      if len(others) > 0:
        residuals = [abs(r_i - r_j) for _, r_j in others]
        D_sum[i] += float(np.mean(residuals))
        D_n[i] += 1
        for j, r_j in others:
          if abs(r_i - r_j) > CONTRA_THRESH:
            c[i] += 1
            c[j] += 1

      cell_reports.setdefault(cell, []).append((i, r_i))
      touched_cells.add(cell)

    mu_c, sigma_c = c.mean(), c.std()
    z = (c - mu_c) / (sigma_c + EPSILON)
    D = np.divide(D_sum, D_n, out=np.zeros_like(D_sum), where=D_n > 0)

    c_norm = c / (t + 1)

    feats = torch.tensor(np.stack([c_norm, z, D], axis=1), dtype=torch.float32)
    alpha = alpha_net(feats)

    step_loss = 0.0
    for cell in touched_cells:
      reports = cell_reports[cell]
      idx = torch.tensor([r for r, _ in reports])
      vals = torch.tensor([v for _, v in reports], dtype=torch.float32)

      a = alpha[idx]
      M = (a * vals).sum() / (a.sum() + EPSILON)
      step_loss = step_loss + (G[cell] - M) ** 2

    step_loss = step_loss / len(touched_cells)

    optimizer.zero_grad()
    step_loss.backward()
    optimizer.step()

    ep_loss += step_loss.item()

  loss_history.append(ep_loss / STEPS)

  if (ep + 1) % 50 == 0:
    print(f"episode {ep+1}/{EPISODES}  avg loss {loss_history[-1]:.5f}")

print("done, final loss:", loss_history[-1])

episode 50/600  avg loss 0.00465
episode 100/600  avg loss 0.00660
episode 150/600  avg loss 0.00402
episode 200/600  avg loss 0.00592
episode 250/600  avg loss 0.00452
episode 300/600  avg loss 0.00651
episode 350/600  avg loss 0.00557
episode 400/600  avg loss 0.00336
episode 450/600  avg loss 0.00463
episode 500/600  avg loss 0.00470
episode 550/600  avg loss 0.00556
episode 600/600  avg loss 0.00466
done, final loss: 0.004656225195006603


In [42]:
c_test = np.array([5, 5, 40, 45])
D_test = np.array([0.05, 0.06, 0.30, 0.32])
z_test = (c_test - c_test.mean()) / (c_test.std() + EPSILON)
c_norm_test = c_test / 100  #consider there are 100 Steps

feats_test = torch.tensor(np.stack([c_norm_test, z_test, D_test], axis=1), dtype=torch.float32)
with torch.no_grad():
    alpha_test = alpha_net(feats_test)

for label, a in zip(["honest-like #1", "honest-like #2", "adversarial-like #1", "adversarial-like #2"], alpha_test):
    print(f"  {label}: alpha = {a.item():.4f}")

  honest-like #1: alpha = 1.0000
  honest-like #2: alpha = 1.0000
  adversarial-like #1: alpha = 0.0771
  adversarial-like #2: alpha = 0.0758
